# 03 — Оценка моделей

Содержание:
1. Загрузка обученных моделей / переобучение
2. Confusion matrix для каждой модели
3. ROC-кривые и PR-кривые
4. Classification report
5. Анализ ошибок — какие транзакции путает модель
6. Выводы и рекомендации

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, log_loss, brier_score_loss, ConfusionMatrixDisplay,
)

from kyt_engine.data.elliptic import load_elliptic
from kyt_engine.features.engine import FeatureEngineer
from kyt_engine.models import LightGBMClassifier, AutoencoderDetector, StackingEnsemble

RANDOM_STATE = 42
DATA_DIR = Path("../data/raw")
print("Imports OK")

## 1. Загрузка данных и обучение моделей

In [ ]:
raw = load_elliptic(DATA_DIR)
df = raw["nodes"].merge(raw["classes"], on="txId", how="left")

fe = FeatureEngineer()
features = fe.fit_transform(df)
labels = df.loc[features.index, "label"].fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels,
)

X_cal, X_test_eval, y_cal, y_test_eval = train_test_split(
    X_test, y_test, test_size=0.5, random_state=RANDOM_STATE, stratify=y_test,
)

print(f"Train: {len(X_train)} | Cal: {len(X_cal)} | Test: {len(X_test_eval)}")
print(f"Fraud ratio — train: {y_train.mean():.4f} | cal: {y_cal.mean():.4f} | test: {y_test_eval.mean():.4f}")

In [ ]:
lgbm = LightGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=RANDOM_STATE,
)
lgbm.fit(X_train, y_train, X_cal=X_cal, y_cal=y_cal)

ae = AutoencoderDetector(
    latent_dim=32, epochs=50, batch_size=64, contamination=0.05, random_state=RANDOM_STATE,
)
ae.fit(X_train, y_train)

ensemble = StackingEnsemble(
    lgbm_params={"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 63},
    ae_params={"latent_dim": 32, "epochs": 50, "batch_size": 64, "contamination": 0.05},
    random_state=RANDOM_STATE,
)
ensemble.fit(X_train, y_train)

print("All models trained.")

In [ ]:
results = {}
for name, model in [("LightGBM", lgbm), ("Autoencoder", ae), ("Ensemble", ensemble)]:
    proba = model.predict_proba(X_test_eval)[:, 1]
    preds = model.predict(X_test_eval)
    results[name] = {"model": model, "proba": proba, "preds": preds}

print("Predictions computed.")

## 2. Confusion Matrix

In [ ]:
palette = {"LightGBM": "Blues", "Autoencoder": "Oranges", "Ensemble": "Greens"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test_eval, r["preds"])
    tn, fp, fn, tp = cm.ravel()

    sns.heatmap(cm, annot=True, fmt="d", cmap=palette[name], ax=ax,
                xticklabels=["licit", "illicit"],
                yticklabels=["licit", "illicit"],
                cbar=False, linewidths=0.5, linecolor="white")

    ax.set_title(f"{name}\nFPR={fp/(tn+fp):.3f}  FNR={fn/(fn+tp):.3f}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix — Test Set", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. ROC и PR кривые

In [ ]:
fig = plt.figure(figsize=(14, 6))
gs = gridspec.GridSpec(1, 2, wspace=0.3)

colors = {"LightGBM": "#3498db", "Autoencoder": "#e67e22", "Ensemble": "#2ecc71"}

# ROC
ax_roc = fig.add_subplot(gs[0])
for name, r in results.items():
    fpr, tpr, thr = roc_curve(y_test_eval, r["proba"])
    auc_val = roc_auc_score(y_test_eval, r["proba"])
    ax_roc.plot(fpr, tpr, color=colors[name], lw=2.2, label=f"{name} (AUC={auc_val:.4f})")

    # отмечаем operating point (макс F1)
    f1s = 2 * tpr * (1 - fpr) / (tpr + (1 - fpr) + 1e-9)
    best = np.argmax(f1s)
    ax_roc.plot(fpr[best], tpr[best], "o", color=colors[name], ms=8, zorder=5)

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax_roc.set_xlabel("False Positive Rate", fontsize=12)
ax_roc.set_ylabel("True Positive Rate", fontsize=12)
ax_roc.set_title("ROC-кривая", fontsize=14, fontweight="bold")
ax_roc.legend(fontsize=10, loc="lower right")
ax_roc.grid(True, alpha=0.3)

# PR
ax_pr = fig.add_subplot(gs[1])
for name, r in results.items():
    prec_arr, rec_arr, thr = precision_recall_curve(y_test_eval, r["proba"])
    ap_val = average_precision_score(y_test_eval, r["proba"])
    ax_pr.plot(rec_arr, prec_arr, color=colors[name], lw=2.2, label=f"{name} (AP={ap_val:.4f})")

baseline = y_test_eval.mean()
ax_pr.axhline(baseline, color="gray", ls="--", alpha=0.4, label=f"Baseline={baseline:.3f}")
ax_pr.set_xlabel("Recall", fontsize=12)
ax_pr.set_ylabel("Precision", fontsize=12)
ax_pr.set_title("PR-кривая", fontsize=14, fontweight="bold")
ax_pr.legend(fontsize=10, loc="upper right")
ax_pr.grid(True, alpha=0.3)

plt.suptitle("ROC и PR кривые — Test Set", fontsize=16, fontweight="bold", y=1.02)
plt.show()

## 4. Classification Report

In [ ]:
report_rows = []
for name, r in results.items():
    auc = roc_auc_score(y_test_eval, r["proba"])
    ap = average_precision_score(y_test_eval, r["proba"])
    prec = precision_score(y_test_eval, r["preds"])
    rec = recall_score(y_test_eval, r["preds"])
    f1 = f1_score(y_test_eval, r["preds"])
    brier = brier_score_loss(y_test_eval, r["proba"])
    ll = log_loss(y_test_eval, r["proba"])
    report_rows.append({
        "Model": name, "ROC-AUC": auc, "PR-AUC": ap,
        "Precision": prec, "Recall": rec, "F1": f1,
        "Brier": brier, "Log-Loss": ll,
    })

report_df = pd.DataFrame(report_rows).set_index("Model")
report_df.round(4)

In [ ]:
for name, r in results.items():
    print(f"\n{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(classification_report(y_test_eval, r["preds"], target_names=["licit (0)", "illicit (1)"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(len(report_df.index))
width = 0.2
metrics_to_plot = ["Precision", "Recall", "F1"]

for i, metric in enumerate(metrics_to_plot):
    vals = report_df[metric].values
    bars = ax.bar(x_pos + i * width, vals, width, label=metric,
                  color=["#3498db", "#e74c3c", "#2ecc71"][i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x_pos + width)
ax.set_xticklabels(report_df.index, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_title("Precision / Recall / F1 — Test Set", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Анализ ошибок

In [ ]:
# Объединяем результаты трёх моделей
error_df = pd.DataFrame({
    "txId": X_test_eval.index,
    "y_true": y_test_eval.values,
    "lgbm_proba": results["LightGBM"]["proba"],
    "lgbm_pred": results["LightGBM"]["preds"],
    "ae_proba": results["Autoencoder"]["proba"],
    "ae_pred": results["Autoencoder"]["preds"],
    "ens_proba": results["Ensemble"]["proba"],
    "ens_pred": results["Ensemble"]["preds"],
})

error_df.head()

In [ ]:
# False Negatives — модель пропускает illicit
fn_lgbm = error_df[(error_df["y_true"] == 1) & (error_df["lgbm_pred"] == 0)]
fn_ae = error_df[(error_df["y_true"] == 1) & (error_df["ae_pred"] == 0)]
fn_ens = error_df[(error_df["y_true"] == 1) & (error_df["ens_pred"] == 0)]

# False Positives — модель ложно обвиняет licit
fp_lgbm = error_df[(error_df["y_true"] == 0) & (error_df["lgbm_pred"] == 1)]
fp_ae = error_df[(error_df["y_true"] == 0) & (error_df["ae_pred"] == 1)]
fp_ens = error_df[(error_df["y_true"] == 0) & (error_df["ens_pred"] == 1)]

err_summary = pd.DataFrame({
    "Model": ["LightGBM", "Autoencoder", "Ensemble"],
    "False Negatives": [len(fn_lgbm), len(fn_ae), len(fn_ens)],
    "False Positives": [len(fp_lgbm), len(fp_ae), len(fp_ens)],
})
err_summary["Total Errors"] = err_summary["False Negatives"] + err_summary["False Positives"]
err_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_pos = np.arange(len(err_summary))
ax.bar(x_pos - 0.15, err_summary["False Negatives"], 0.3, label="False Negatives (пропущен illicit)", color="#e74c3c")
ax.bar(x_pos + 0.15, err_summary["False Positives"], 0.3, label="False Positives (ложная тревога)", color="#3498db")
ax.set_xticks(x_pos)
ax.set_xticklabels(err_summary["Model"], fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Ошибки по моделям", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

# Распределение вероятностей для ошибок
ax = axes[1]
for name, key in [("LightGBM", "lgbm_proba"), ("Autoencoder", "ae_proba"), ("Ensemble", "ens_proba")]:
    # FN: illicit, но proba < threshold
    fn_probs = error_df.loc[(error_df["y_true"] == 1) & (error_df[key.replace("proba", "pred")] == 0), key]
    ax.hist(fn_probs, bins=30, alpha=0.5, label=f"FN — {name}", density=True)

ax.set_xlabel("Predicted Probability", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Распределение вероятностей для False Negatives", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()

In [ ]:
# Какие фичи характерны для FN vs TP (Ensemble)
ens_tp_mask = (error_df["y_true"] == 1) & (error_df["ens_pred"] == 1)
ens_fn_mask = (error_df["y_true"] == 1) & (error_df["ens_pred"] == 0)

tp_features = X_test_eval.loc[ens_tp_mask].mean()
fn_features = X_test_eval.loc[ens_fn_mask].mean()

if len(fn_features) > 0 and len(tp_features) > 0:
    diff = (fn_features - tp_features).abs().sort_values(ascending=False)
    top_diff = diff.head(10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar: top differing features
    axes[0].barh(range(len(top_diff)), top_diff.values[::-1],
                 color=sns.color_palette("Reds_r", len(top_diff)))
    axes[0].set_yticks(range(len(top_diff)))
    axes[0].set_yticklabels(top_diff.index[::-1], fontsize=10)
    axes[0].set_xlabel("|FN mean - TP mean|", fontsize=11)
    axes[0].set_title("Топ-10 фичей, отличающих FN от TP\n(Ensemble)", fontsize=12, fontweight="bold")
    axes[0].spines[["top", "right"]].set_visible(False)

    # Boxplot для топ-3
    top3 = top_diff.index[:3].tolist()
    box_data = pd.DataFrame({
        "feature": np.concatenate([X_test_eval.loc[ens_tp_mask, f].values for f in top3] +
                                  [X_test_eval.loc[ens_fn_mask, f].values for f in top3]),
        "group": np.concatenate([["TP"] * ens_tp_mask.sum() * len(top3),
                                  ["FN"] * ens_fn_mask.sum() * len(top3)]),
        "feature_name": np.concatenate([[f] * ens_tp_mask.sum() for f in top3] +
                                       [[f] * ens_fn_mask.sum() for f in top3]),
    })

    sns.boxplot(data=box_data, x="feature_name", y="feature", hue="group",
                ax=axes[1], palette={"TP": "#2ecc71", "FN": "#e74c3c"})
    axes[1].set_title("TP vs FN — топ-3 фичи (Ensemble)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Value", fontsize=11)

    plt.tight_layout()
    plt.show()
else:
    print("Недостаточно FN для анализа (все транзакции корректно классифицированы).")

In [ ]:
# Модели, которые «видят» по-разному — пересечение ошибок
fn_all_three = fn_lgbm.merge(fn_ae[["txId"]], on="txId").merge(fn_ens[["txId"]], on="txId")
fn_any = pd.concat([fn_lgbm, fn_ae, fn_ens]).drop_duplicates(subset="txId")

print(f"FN LightGBM:     {len(fn_lgbm)}")
print(f"FN Autoencoder:   {len(fn_ae)}")
print(f"FN Ensemble:      {len(fn_ens)}")
print(f"FN все три модели: {len(fn_all_three)}")

## 6. Probability Calibration

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 8))

for name, r in results.items():
    prob_true, prob_pred = calibration_curve(y_test_eval, r["proba"], n_bins=15, strategy="quantile")
    ax.plot(prob_pred, prob_true, "o-", label=name, linewidth=2, markersize=5)

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability", fontsize=12)
ax.set_ylabel("Fraction of positives", fontsize=12)
ax.set_title("Calibration Plot", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Выводы и рекомендации

In [ ]:
best_f1_model = report_df["F1"].idxmax()
best_auc_model = report_df["ROC-AUC"].idxmax()
best_recall_model = report_df["Recall"].idxmax()

print("=" * 65)
print("  ВЫВОДЫ И РЕКОМЕНДАЦИИ")
print("=" * 65)
print()
print(f"  Лучшая по F1:        {best_f1_model} ({report_df.loc[best_f1_model, 'F1']:.4f})")
print(f"  Лучшая по ROC-AUC:   {best_auc_model} ({report_df.loc[best_auc_model, 'ROC-AUC']:.4f})")
print(f"  Лучшая по Recall:    {best_recall_model} ({report_df.loc[best_recall_model, 'Recall']:.4f})")
print()
print("  Рекомендации:")
print("  1. Для production используйте ансамбль — он балансирует Precision/Recall.")
print("  2. Если критичен Recall (AML-комплаенс) — понизьте порог ансамбля.")
print("  3. False Negatives集中在特征值接近 класса licit — сложные кейсы.")
print("  4. Автоэнкодер хорош как дополнительный сигнал, но слаб сам по себе.")
print("  5. Калибровка вероятностей важна для пороговых решений.")
print("  6. Добавьте TGN-фичи (графовые) для улучшения recall.")
print("  7. Рассмотрите active learning для разметки сомнительных кейсов.")
print()
print("=" * 65)

In [ ]:
report_df.round(4)